# CAREGIVER - Silver Layer

This notebook validates the MIMIC-IV `caregiver` Bronze table before storing it in the Silver layer.

`CAREGIVER` is a simple reference table containing the de-identified caregiver IDs used in ICU event data. It does not contain names, roles or other staff details, so its main purpose is to provide a valid reference for `caregiver_id`.

The Silver check focuses on confirming that `caregiver_id` remains complete and unique.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

# Load Bronze table
caregiver = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.caregiver_raw")

print(f"Rows: {caregiver.count():,}")
print(f"Columns: {len(caregiver.columns)}")

caregiver.printSchema()

In [0]:
# Validate caregiver_id
total = caregiver.count()
unique_ids = caregiver.select("caregiver_id").distinct().count()
null_ids = caregiver.filter(F.col("caregiver_id").isNull()).count()

print(f"Rows: {total:,}")
print(f"Unique caregiver_id: {unique_ids:,}")
print(f"Null caregiver_id: {null_ids:,}")
print(f"Duplicate caregiver_id: {total - unique_ids:,}")

### Silver Result

`CAREGIVER` is already clean at the identifier level, with no missing or duplicate `caregiver_id` values.

Because the table only contains the de-identified caregiver identifier, no additional cleaning is required before storing it in Silver.

In [0]:
# Save CAREGIVER Silver table
(
    caregiver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SILVER_SCHEMA}.caregiver")
)

print("Created: workspace.silver.caregiver")

In [0]:
bronze_count = spark.table("workspace.bronze.caregiver_raw").count()
silver_count = spark.table("workspace.silver.caregiver").count()

print(f"Bronze: {bronze_count:,}")
print(f"Silver: {silver_count:,}")
print(f"Match: {bronze_count == silver_count}")

### Silver Result

`CAREGIVER` was successfully written to the Silver layer with all records preserved.

The table required no additional transformation because `caregiver_id` is already complete and unique.